In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv("D:\\Github-Contributer-predictor\\data\\raw\\pandas_prs.csv")
df.columns

df.describe(include='all')


df.info(memory_usage='deep')


C:\Users\DRISTI\AppData\Local\Temp\ipykernel_27444\600003852.py:1: DtypeWarning: Columns (31,341,345,347,348,350,351,352,353,354,355,356,357,358,359,360,361,362,363) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("D:\\Github-Contributer-predictor\\data\\raw\\pandas_prs.csv")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12100 entries, 0 to 12099
Columns: 364 entries, url to assignee.site_admin
dtypes: bool(19), float64(31), int64(14), object(300)
memory usage: 336.3 MB


In [3]:
def audit_dataframe(df):
    audit_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        '% Missing': (df.isnull().mean()) * 100,
        'Unique Values': df.nunique()
    })
    
    return audit_df.sort_values(by='% Missing', ascending=False).round(2)

print(audit_dataframe(df))

                              Data Type  ...  Unique Values
assignee                        float64  ...              0
auto_merge                      float64  ...              0
milestone.creator.gravatar_id   float64  ...              0
user.gravatar_id                float64  ...              0
base.repo.mirror_url            float64  ...              0
...                                 ...  ...            ...
diff_url                         object  ...          12100
html_url                         object  ...          12100
node_id                          object  ...          12100
id                                int64  ...          12100
url                              object  ...          12100

[364 rows x 4 columns]


In [4]:
print("columns before drop:",df.shape[1])
df=df.dropna(axis=1,how='all')
print("columns after:",df.shape[1])

columns before drop: 364
columns after: 350


In [5]:
constant_cols = df.columns[df.nunique(dropna=False) == 1]

print(f"Constant columns: {len(constant_cols)}")
print(constant_cols)

Constant columns: 123
Index(['requested_teams', 'review_comment_url', 'user.user_view_type',
       'base.user.login', 'base.user.id', 'base.user.node_id',
       'base.user.avatar_url', 'base.user.url', 'base.user.html_url',
       'base.user.followers_url',
       ...
       'base.repo.web_commit_signoff_required', 'base.repo.has_pull_requests',
       'base.repo.pull_request_creation_policy', 'base.repo.topics',
       'base.repo.visibility', 'base.repo.forks', 'base.repo.open_issues',
       'base.repo.watchers', 'base.repo.default_branch',
       '_links.review_comment.href'],
      dtype='object', length=123)


In [6]:
url_cols = [col for col in df.columns if "url" in col.lower()]
df = df.drop(columns=url_cols)
df.shape[1]

167

In [7]:
constant_cols = df.columns[df.nunique(dropna=False) == 1]
print(len(constant_cols))

55


In [8]:
df = df.drop(columns=constant_cols)

In [9]:
df.shape[1]

112

In [10]:
df["user.id"]

0         8078968
1         8078968
2         8078968
3         8078968
4         8078968
           ...   
12095    33491632
12096    39504233
12097    39504233
12098    61934744
12099    61934744
Name: user.id, Length: 12100, dtype: int64

In [11]:
def audit_dataframe(df):
    audit_df = pd.DataFrame({
        'Data Type': df.dtypes,
        'Missing Values': df.isnull().sum(),
        '% Missing': (df.isnull().mean()) * 100,
        'Unique Values': df.nunique()
    })
    return audit_df.sort_values(by='% Missing', ascending=False).round(2)

print(audit_dataframe(df))

                            Data Type  Missing Values  % Missing  Unique Values
active_lock_reason             object           12097      99.98              2
assignee.node_id               object           12092      99.93              4
assignee.user_view_type        object           12092      99.93              1
assignee.site_admin            object           12092      99.93              1
assignee.type                  object           12092      99.93              1
...                               ...             ...        ...            ...
_links.issue.href              object               0       0.00          12100
_links.comments.href           object               0       0.00          12100
_links.review_comments.href    object               0       0.00          12100
_links.commits.href            object               0       0.00          12100
_links.statuses.href           object               0       0.00          12070

[112 rows x 4 columns]


In [12]:

df['body'] = df['body'].fillna("no_description")

df['title'] = df['title'].fillna("no_title")

In [13]:
date_cols=[col for col in df.columns if "_at" in col.lower()]
print(date_cols)
for col in date_cols:
    df[col]=pd.to_datetime(df[col])
print(df[date_cols].dtypes)

['created_at', 'updated_at', 'closed_at', 'merged_at', 'head.repo.created_at', 'head.repo.updated_at', 'head.repo.pushed_at', 'milestone.created_at', 'milestone.updated_at', 'milestone.closed_at']
created_at              datetime64[ns, UTC]
updated_at              datetime64[ns, UTC]
closed_at               datetime64[ns, UTC]
merged_at               datetime64[ns, UTC]
head.repo.created_at    datetime64[ns, UTC]
head.repo.updated_at    datetime64[ns, UTC]
head.repo.pushed_at     datetime64[ns, UTC]
milestone.created_at    datetime64[ns, UTC]
milestone.updated_at    datetime64[ns, UTC]
milestone.closed_at     datetime64[ns, UTC]
dtype: object


In [14]:
df = df.sort_values(by=['user.login', 'created_at'])
df = df.reset_index(drop=True)

In [15]:
missing_percent = df.isna().mean() * 100

high_missing_cols = missing_percent[missing_percent > 95].index

print(f"Dropping {len(high_missing_cols)} columns")
print(high_missing_cols.tolist())

df = df.drop(columns=high_missing_cols)


Dropping 7 columns
['active_lock_reason', 'assignee.login', 'assignee.id', 'assignee.node_id', 'assignee.type', 'assignee.user_view_type', 'assignee.site_admin']


In [16]:
[col for col in df.columns if "login" in col.lower()]
contributers=df["user.login"]
user=contributers[0]
user_df=df[df["user.login"]==user]
print(user_df[["user.login", "created_at", "merged_at", "title"]])

  user.login  ...                                              title
0     0x3vAD  ...                                        Ignore this
1     0x3vAD  ...  BUG: Fix assert_frame_equal with check_dtype=F...

[2 rows x 4 columns]


In [17]:
df["author_association"].value_counts()

author_association
MEMBER         5436
CONTRIBUTOR    5285
NONE           1379
Name: count, dtype: int64

In [18]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder()


Dataset Design

In [31]:
def build_dataset_for_N(df, N, W_days=180):
    
    dataset_max_date = df['created_at'].max()
    
    df = df.sort_values(['user.id', 'created_at']).copy()
    df['pr_rank'] = df.groupby('user.id').cumcount() + 1
    
    pr_counts = df.groupby('user.id').size()
    eligible_users = pr_counts[pr_counts >= N].index
    df_eligible = df[df['user.id'].isin(eligible_users)].copy()
    
    obs = df_eligible[df_eligible['pr_rank'] <= N]
    future = df_eligible[df_eligible['pr_rank'] > N]
    
    cutoffs = obs[obs['pr_rank'] == N][['user.id', 'created_at']].rename(
        columns={'created_at': 'cutoff_date'}
    )
    
    W = pd.Timedelta(days=W_days)
    cutoffs['outcome_end'] = cutoffs['cutoff_date'] + W
    cutoffs['is_censored'] = cutoffs['outcome_end'] > dataset_max_date
    
    labels_df = cutoffs[~cutoffs['is_censored']].copy()
    
    future_tracked = future.merge(labels_df, on='user.id', how='inner')
    
   
    active_in_window = future_tracked[
        (future_tracked['created_at'] > future_tracked['cutoff_date']) & 
        (future_tracked['created_at'] <= future_tracked['outcome_end'])
    ]
    
    retained_users = active_in_window['user.id'].unique()
    labels_df['label'] = labels_df['user.id'].isin(retained_users).astype(int)
    
    labels_df = labels_df[['user.id', 'label']]
    obs["days_since_prev_pr"] = (
    obs.groupby("user.id")["created_at"]
       .diff()
       .dt.days
)
    features = obs.groupby("user.id").agg(
    n_obs_prs=("pr_rank", "count"),
    frac_merged=("merged_at", lambda x: x.notna().mean()),
    first_pr_date=("created_at", "min"),
    last_obs_pr_date=("created_at", "max"),
    author_association=("author_association", "last"),
    avg_days_between_prs=("days_since_prev_pr", "mean")
).reset_index()
    
    features['days_to_reach_Nth_pr'] = (
        features['last_obs_pr_date'] - features['first_pr_date']
    ).dt.days
    
    features = features.drop(columns=['first_pr_date', 'last_obs_pr_date'])
    
    final_df = features.merge(labels_df, on='user.id', how='inner')
    return final_df

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score



candidate_N = [2, 3,4,5,10]

results = []

best_score = -1
best_N = None
best_splits = None

for N in candidate_N:

    # Build dataset
    dataset = build_dataset_for_N(df, N)

    X = dataset.drop(columns=["user.id", "label"])
    y = dataset["label"]

    # Train (80%) and Temp (20%)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Validation (10%) and Test (10%)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    # Train model
    
    model = LogisticRegression(class_weight="balanced")
    model.fit(X_train, y_train)

    # Evaluate on validation set
    y_val_pred = model.predict(X_val)
    val_score = f1_score(y_val, y_val_pred)

    print(f"N = {N}, Validation F1 = {val_score:.4f}")
    print(f"Eligible contributors: {len(dataset)}")
    print(dataset["label"].value_counts())

    results.append({
        "N": N,
        "validation_score": val_score
    })

results = pd.DataFrame(results)

print("\nBest N:", best_N)
print("Best Validation F1:", best_score)



C:\Users\DRISTI\AppData\Local\Temp\ipykernel_27444\3551542974.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obs["days_since_prev_pr"] = (


ValueError: could not convert string to float: 'CONTRIBUTOR'

In [30]:
best_N = 3
final_df=build_dataset_for_N(df,3)
X = final_df.drop(columns=["user.id", "label"])
y = final_df["label"]

# Train/Validation/Test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# Combine train and validation
X_train_final = pd.concat([X_train, X_val], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Train final model
final_model = LogisticRegression(class_weight="balanced")
final_model.fit(X_train_final, y_train_final)
y_test_pred = final_model.predict(X_test)

test_f1 = f1_score(y_test, y_test_pred)

print(f"Final Test F1 Score: {test_f1:.4f}")

Final Test F1 Score: 0.6829
